In [ ]:
import os
from getpass import getpass
import requests

# ============================================================
# SETTINGS
# ============================================================

# Fill in your own Copernicus Data Space Ecosystem account
# (free registration at https://dataspace.copernicus.eu).
# Leave these empty to be asked for them when the cell runs.
USERNAME = ""
PASSWORD = ""
if not USERNAME:
    USERNAME = input("Copernicus username (e-mail): ")
if not PASSWORD:
    PASSWORD = getpass("Copernicus password: ")

download_folder = "Output/s3"

# Approximate Bordeaux region
# Coordinates are longitude, latitude (EPSG:4326)
lon_min = -0.80
lon_max = -0.35
lat_min = 44.70
lat_max = 45.05

# Only 26 August (start inclusive, end exclusive)
start_date = "2026-08-26T00:00:00.000Z"
end_date   = "2026-08-27T00:00:00.000Z"

os.makedirs(download_folder, exist_ok=True)


# ============================================================
# DEFINE BORDEAUX AOI
# ============================================================

aoi = (
    f"POLYGON(("
    f"{lon_min} {lat_min},"
    f"{lon_max} {lat_min},"
    f"{lon_max} {lat_max},"
    f"{lon_min} {lat_max},"
    f"{lon_min} {lat_min}"
    f"))"
)


# ============================================================
# SEARCH FOR SENTINEL-3 SLSTR FRP PRODUCTS
# ============================================================

catalogue_url = (
    "https://catalogue.dataspace.copernicus.eu/"
    "odata/v1/Products"
)

filter_string = (
    "Collection/Name eq 'SENTINEL-3' "
    "and Attributes/OData.CSC.StringAttribute/any("
    "att:att/Name eq 'productType' "
    "and att/OData.CSC.StringAttribute/Value eq 'SL_2_LST___'"
    ") "
    f"and OData.CSC.Intersects("
    f"area=geography'SRID=4326;{aoi}'"
    f") "
    f"and ContentDate/Start ge {start_date} "
    f"and ContentDate/Start lt {end_date}"
)

params = {
    "$filter": filter_string,
    "$orderby": "ContentDate/Start asc",
    "$top": 1000
}

response = requests.get(
    catalogue_url,
    params=params,
    timeout=60
)

response.raise_for_status()

data = response.json()
products = data["value"]


# ============================================================
# SHOW PRODUCTS FOUND
# ============================================================

products = [
    p for p in products
    if "_NT_" in p["Name"]
]

print(f"Found {len(products)} NTC FRP products:\n")

for i, product in enumerate(products, start=1):
    print(f"{i:3d}: {product['Name']}")


In [ ]:
# ============================================================
# GET ACCESS TOKEN
# ============================================================

token_url = (
    "https://identity.dataspace.copernicus.eu/"
    "auth/realms/CDSE/protocol/openid-connect/token"
)

token_data = {
    "client_id": "cdse-public",
    "username": USERNAME,
    "password": PASSWORD,
    "grant_type": "password",
}

token_response = requests.post(
    token_url,
    data=token_data,
    timeout=60
)

token_response.raise_for_status()

access_token = token_response.json()["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}"
}


# ============================================================
# DOWNLOAD PRODUCTS
# ============================================================

for i, product in enumerate(products, start=1):

    product_id = product["Id"]
    product_name = product["Name"]

    output_file = os.path.join(
        download_folder,
        product_name + ".zip"
    )

    # Skip products that were already downloaded
    if os.path.exists(output_file):
        print(f"[{i}/{len(products)}] Already exists: {product_name}")
        continue

    download_url = (
        "https://download.dataspace.copernicus.eu/"
        f"odata/v1/Products({product_id})/$value"
    )

    print(f"[{i}/{len(products)}] Downloading:")
    print(product_name)

    with requests.get(
        download_url,
        headers=headers,
        stream=True,
        timeout=300
    ) as r:

        r.raise_for_status()

        with open(output_file, "wb") as f:

            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)

    print("Finished\n")


print("All downloads finished.")

In [ ]:
import re
import tempfile
import zipfile
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

s3a_folder = Path("Output/s3")
s3a_files = sorted(s3a_folder.glob("S3A_SL_2_LST____20260826T1035*.zip"))

if not s3a_files:
    raise FileNotFoundError(f"No S3A ZIP products found in {s3a_folder}")

# Load every wider-area LST raster; the next cell plots and saves each one separately.
map_extent = [-1.25, -0.05, 44.35, 45.35]
nearby_cities = {
    "Bordeaux": (-0.5792, 44.8378),
    "Libourne": (-0.2410, 44.9150),
    "Arcachon": (-1.1740, 44.6580),
    "Langon": (-0.2500, 44.5530),
}


def find_variable(dataset, candidates, label):
    for name in dataset.variables:
        if name.lower() in candidates:
            return dataset[name]

    raise KeyError(
        f"Could not find {label} variable. Available variables: {list(dataset.variables)}"
    )


def extract_lst_grid(zip_path):
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(zip_path) as archive:
            members = archive.namelist()
            lst_member = next(member for member in members if member.endswith("/LST_in.nc"))
            geodetic_member = next(member for member in members if member.endswith("/geodetic_in.nc"))
            archive.extract(lst_member, temp_dir)
            archive.extract(geodetic_member, temp_dir)

        with xr.open_dataset(Path(temp_dir) / lst_member) as lst_dataset, xr.open_dataset(
            Path(temp_dir) / geodetic_member
        ) as geodetic_dataset:
            land_surface_temperature = find_variable(
                lst_dataset,
                ["lst"],
                "land surface temperature",
            )
            latitude = find_variable(geodetic_dataset, ["latitude_in"], "latitude")
            longitude = find_variable(geodetic_dataset, ["longitude_in"], "longitude")
            land_surface_temperature, latitude, longitude = xr.broadcast(
                land_surface_temperature,
                latitude,
                longitude,
            )
            return (
                land_surface_temperature.values.astype(float),
                latitude.values.astype(float),
                longitude.values.astype(float),
            )


s3a_images = []
for image_path in s3a_files:
    image_date = re.search(r"S3A_SL_2_LST____(\d{8})T(\d{6})", image_path.name)
    if image_date:
        lst_grid, latitude_grid, longitude_grid = extract_lst_grid(image_path)
        in_map_area = (
            np.isfinite(latitude_grid)
            & np.isfinite(longitude_grid)
            & (longitude_grid >= map_extent[0])
            & (longitude_grid <= map_extent[1])
            & (latitude_grid >= map_extent[2])
            & (latitude_grid <= map_extent[3])
        )
        row_indices, column_indices = np.where(in_map_area)
        if len(row_indices) == 0:
            continue

        row_slice = slice(row_indices.min(), row_indices.max() + 1)
        column_slice = slice(column_indices.min(), column_indices.max() + 1)
        lst_grid = lst_grid[row_slice, column_slice]
        latitude_grid = latitude_grid[row_slice, column_slice]
        longitude_grid = longitude_grid[row_slice, column_slice]
        valid_lst = (
            in_map_area[row_slice, column_slice]
            & np.isfinite(lst_grid)
            & (lst_grid > 0)
        )
        lst_grid = np.where(valid_lst, lst_grid, np.nan)
        s3a_images.append(
            {
                "path": image_path,
                "date": image_date.group(1),
                "time": image_date.group(2),
                "lst": lst_grid,
                "latitude": latitude_grid,
                "longitude": longitude_grid,
            }
        )

if not s3a_images:
    raise ValueError("No S3A LST images intersect the selected map area")

finite_lst = np.concatenate(
    [image["lst"][np.isfinite(image["lst"])] for image in s3a_images]
)
if finite_lst.size == 0:
    raise ValueError("No valid LST pixels were found in the selected map area")

# One shared colour scale so all images are directly comparable
lst_min = float(finite_lst.min())
lst_max = float(finite_lst.max())
if lst_min == lst_max:
    lst_max = lst_min + 1.0

map_projection = ccrs.PlateCarree()
print(f"Wider-area S3A LST images available: {len(s3a_images)}")

In [ ]:
figure_folder = Path("Output/figures")
figure_folder.mkdir(parents=True, exist_ok=True)

# Only plot the 10:35 UTC overpass, even if s3a_images still holds other products
selected_images = [image for image in s3a_images if image["time"].startswith("1035")]

for image in selected_images:
    figure, map_ax = plt.subplots(
        figsize=(12, 10),
        subplot_kw={"projection": map_projection},
        constrained_layout=True,
    )
    map_ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf2", zorder=0)
    map_ax.add_feature(cfeature.LAND, facecolor="#eeeeea", zorder=0)
    map_ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#444444", zorder=3)
    map_ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="#666666", zorder=3)
    map_ax.set_extent(map_extent, crs=map_projection)

    # Colour scale from this image only, so the 21:58 UTC product never affects it
    image_lst = image["lst"][np.isfinite(image["lst"])]
    raster = map_ax.pcolormesh(
        image["longitude"],
        image["latitude"],
        image["lst"],
        cmap="inferno",
        vmin=float(image_lst.min()),
        vmax=float(image_lst.max()),
        shading="auto",
        alpha=0.62,
        transform=map_projection,
        zorder=2,
    )

    city_longitudes = [coordinates[0] for coordinates in nearby_cities.values()]
    city_latitudes = [coordinates[1] for coordinates in nearby_cities.values()]
    map_ax.scatter(
        city_longitudes,
        city_latitudes,
        c="red",
        s=50,
        edgecolors="white",
        linewidths=1.2,
        transform=map_projection,
        zorder=4,
    )
    for city_name, (longitude, latitude) in nearby_cities.items():
        map_ax.text(
            longitude + 0.02,
            latitude + 0.02,
            city_name,
            fontsize=18 if city_name == "Bordeaux" else 14,
            color="black",
            ha="left",
            va="bottom",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 2},
            transform=map_projection,
            zorder=5,
        )

    map_ax.set_title(f"{image['date']} | {image['time'][:2]}:{image['time'][2:4]} UTC", fontsize=16)
    map_ax.set_xlabel("Longitude", fontsize=16)
    map_ax.set_ylabel("Latitude", fontsize=16)
    gridlines = map_ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        alpha=0.5,
        linestyle="--",
    )
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {"size": 16}
    gridlines.ylabel_style = {"size": 16}

    # Colorbar axes in map-axes coordinates, so it matches the map height exactly
    colorbar_ax = map_ax.inset_axes([1.03, 0, 0.04, 1])
    colorbar = figure.colorbar(raster, cax=colorbar_ax, orientation="vertical")
    colorbar.set_label("Oppervlakte temperatuur (K)", fontsize=22)
    colorbar.ax.tick_params(labelsize=22)

    figure.savefig(
        figure_folder / f"LST_{image['date']}_{image['time']}.svg",
        format="svg",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

print(f"Saved {len(selected_images)} SVG figures to {figure_folder}")

In [ ]:
from IPython.display import display

# Child-friendly version of the figure above: no numbers on the colorbar,
# just "Laag" at the bottom, "Hoog" at the top and an arrow from low to high.
low_label, high_label = "Laag", "Hoog"

# Remove earlier additions so re-running this cell does not stack them
for artist in [text for text in colorbar.ax.texts if text.get_gid() == "kids_colorbar"]:
    artist.remove()

colorbar.set_ticks([])
colorbar.ax.text(0.5, -0.01, low_label, transform=colorbar.ax.transAxes,
                 ha="center", va="top", fontsize=22, gid="kids_colorbar")
colorbar.ax.text(0.5, 1.01, high_label, transform=colorbar.ax.transAxes,
                 ha="center", va="bottom", fontsize=22, gid="kids_colorbar")
colorbar.ax.annotate(
    "",
    xy=(2.0, 0.97),
    xytext=(2.0, 0.03),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "linewidth": 3, "color": "black", "mutation_scale": 30},
    annotation_clip=False,
    gid="kids_colorbar",
)
colorbar.ax.yaxis.labelpad = 45  # keep the colorbar title clear of the arrow

kids_svg = figure_folder / f"LST_{image['date']}_{image['time']}_kids.svg"
figure.savefig(kids_svg, format="svg", bbox_inches="tight")
display(figure)
print(f"Saved {kids_svg}")